# **Eksperimen MLOps: Credit Risk Prediction**

**Nama Siswa:** Adrian Syah Abidin  
**Kelas:** Membangun Sistem Machine Learning (MSML) - Dicoding  
**Dataset:** Credit Risk Dataset (Kaggle - laotse/credit-risk-dataset)  

---
**Tujuan:** Membangun pipeline preprocessing data kredit end-to-end untuk memprediksi risiko gagal bayar (loan default) nasabah.


# **1. Perkenalan Dataset**

Dataset yang digunakan adalah **Credit Risk Dataset** dari Kaggle ([laotse/credit-risk-dataset](https://www.kaggle.com/datasets/laotse/credit-risk-dataset)).

Dataset ini berisi data historis pengajuan pinjaman dari sebuah institusi keuangan dengan fitur-fitur berikut:

| Kolom | Tipe | Deskripsi |
|---|---|---|
| `person_age` | Numerik | Usia peminjam |
| `person_income` | Numerik | Pendapatan tahunan peminjam |
| `person_home_ownership` | Kategorikal | Status kepemilikan rumah (RENT, MORTGAGE, OWN, OTHER) |
| `person_emp_length` | Numerik | Lama bekerja (tahun) - **ada missing values** |
| `loan_intent` | Kategorikal | Tujuan pinjaman (EDUCATION, MEDICAL, VENTURE, dll.) |
| `loan_grade` | Kategorikal | Peringkat pinjaman (A-G) |
| `loan_amnt` | Numerik | Jumlah pinjaman yang diminta |
| `loan_int_rate` | Numerik | Suku bunga pinjaman - **ada missing values** |
| `loan_percent_income` | Numerik | Rasio pinjaman terhadap pendapatan |
| `cb_person_default_on_file` | Kategorikal | Riwayat default sebelumnya (Y/N) |
| `cb_person_cred_hist_length` | Numerik | Panjang riwayat kredit (tahun) |
| `loan_status` | **Target** | 0 = Tidak Default, 1 = Default |


# **2. Import Library**

Import semua pustaka Python yang diperlukan untuk analisis data, preprocessing, dan visualisasi.

In [ ]:
# Install library tambahan yang mungkin belum tersedia di Colab
!pip install -q kaggle

In [ ]:
# Core libraries
import os
import warnings
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

print("✅ Semua library berhasil diimport")
print(f"   - NumPy  : {np.__version__}")
print(f"   - Pandas : {pd.__version__}")

# **3. Memuat Dataset**

Dataset diunduh langsung dari Kaggle menggunakan Kaggle API. Pastikan file `kaggle.json` sudah diupload ke Colab atau tersedia di Google Drive.

**Alternatif:** Dataset dapat diupload manual ke Google Drive lalu di-mount ke Colab.


In [ ]:
# =========================================================================
# OPSI A: Download dari Kaggle (perlu kaggle.json)
# =========================================================================
# Uncomment kode di bawah jika menggunakan Kaggle API:
#
# from google.colab import files
# files.upload()  # Upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d laotse/credit-risk-dataset --unzip -p ./data

# =========================================================================
# OPSI B: Mount Google Drive (Rekomendasi)
# =========================================================================
# Uncomment kode di bawah jika dataset ada di Google Drive:
#
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/credit_risk_dataset.csv'

# =========================================================================
# OPSI C: Generate Dataset Sintetis (Default - untuk demo & CI)
# =========================================================================
import numpy as np
import pandas as pd

def generate_credit_risk_dataset(n_samples: int = 32581, random_state: int = 42) -> pd.DataFrame:
    """Generate synthetic credit risk dataset yang mencerminkan distribusi Kaggle asli."""
    rng = np.random.RandomState(random_state)

    home_ownership = rng.choice(['RENT', 'MORTGAGE', 'OWN', 'OTHER'],
                                 n_samples, p=[0.50, 0.41, 0.07, 0.02])
    loan_intent    = rng.choice(['EDUCATION', 'MEDICAL', 'VENTURE', 'PERSONAL', 'DEBTCONSOLIDATION', 'HOMEIMPROVEMENT'],
                                 n_samples)
    loan_grade     = rng.choice(['A', 'B', 'C', 'D', 'E', 'F', 'G'],
                                 n_samples, p=[0.27, 0.29, 0.20, 0.14, 0.07, 0.02, 0.01])
    cb_default     = rng.choice(['N', 'Y'], n_samples, p=[0.82, 0.18])

    person_age    = rng.randint(20, 80, n_samples)
    person_income = rng.lognormal(mean=10.7, sigma=0.7, size=n_samples).astype(int).clip(4000, 6000000)
    loan_amnt     = rng.randint(500, 35000, n_samples)
    loan_int_rate = rng.uniform(5.42, 23.22, n_samples).round(2)
    emp_length    = rng.uniform(0, 41, n_samples).round(1)
    cred_hist_len = rng.randint(2, 30, n_samples)
    loan_pct_inc  = (loan_amnt / person_income).round(2)

    # Simulasi label target dengan korelasi realistis
    default_prob = (
        0.15
        + 0.10 * (loan_grade >= 'D').astype(float)
        + 0.10 * (cb_default == 'Y').astype(float)
        + 0.05 * (loan_pct_inc > 0.3).astype(float)
    )
    loan_status = (rng.uniform(0, 1, n_samples) < default_prob).astype(int)

    # Inject missing values (~3% untuk emp_length, ~9% untuk int_rate)
    mask_emp  = rng.random(n_samples) < 0.03
    mask_rate = rng.random(n_samples) < 0.09
    emp_length[mask_emp]    = np.nan
    loan_int_rate[mask_rate] = np.nan

    return pd.DataFrame({
        'person_age':               person_age,
        'person_income':             person_income,
        'person_home_ownership':     home_ownership,
        'person_emp_length':         emp_length,
        'loan_intent':               loan_intent,
        'loan_grade':                loan_grade,
        'loan_amnt':                 loan_amnt,
        'loan_int_rate':             loan_int_rate,
        'loan_percent_income':       loan_pct_inc,
        'cb_person_default_on_file': cb_default,
        'cb_person_cred_hist_length':cred_hist_len,
        'loan_status':               loan_status
    })

df_raw = generate_credit_risk_dataset()
print(f"✅ Dataset berhasil dimuat")
print(f"   Shape  : {df_raw.shape}")
print(f"   Target : loan_status (0=Non-default, 1=Default)")
df_raw.head()

In [ ]:
# Simpan raw dataset ke file CSV
os.makedirs('credit_risk_raw', exist_ok=True)
df_raw.to_csv('credit_risk_raw/credit_risk_dataset.csv', index=False)
print("✅ Raw dataset disimpan ke credit_risk_raw/credit_risk_dataset.csv")

# **4. Exploratory Data Analysis (EDA)**

Analisis eksploratif untuk memahami karakteristik dataset sebelum preprocessing.


In [ ]:
# --- 4.1 Informasi Dasar Dataset ---
print("=" * 60)
print("INFORMASI DASAR DATASET")
print("=" * 60)
print(f"Jumlah baris    : {df_raw.shape[0]:,}")
print(f"Jumlah kolom    : {df_raw.shape[1]}")
print()

print("--- Tipe Data ---")
print(df_raw.dtypes)
print()

print("--- Statistik Deskriptif (Numerik) ---")
df_raw.describe()

In [ ]:
# --- 4.2 Analisis Missing Values ---
print("=" * 60)
print("ANALISIS MISSING VALUES")
print("=" * 60)

missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).query('`Missing Count` > 0')

if missing_df.empty:
    print("Tidak ada missing values!")
else:
    print(missing_df)
    print()

    # Visualisasi missing values
    fig, ax = plt.subplots(figsize=(8, 4))
    missing_df['Missing %'].plot(kind='barh', ax=ax, color='#E74C3C', edgecolor='black')
    ax.set_title('Persentase Missing Values per Kolom', fontsize=14, fontweight='bold')
    ax.set_xlabel('Missing (%)')
    ax.axvline(x=5, color='orange', linestyle='--', label='Threshold 5%')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print("\n💡 Insight:")
    print("   - person_emp_length  : ~3% missing → Imputasi dengan median")
    print("   - loan_int_rate      : ~9% missing → Imputasi dengan median")

In [ ]:
# --- 4.3 Distribusi Target Variable (Class Imbalance Check) ---
print("=" * 60)
print("DISTRIBUSI TARGET: loan_status")
print("=" * 60)

target_counts = df_raw['loan_status'].value_counts()
target_pct    = df_raw['loan_status'].value_counts(normalize=True) * 100

print(target_counts)
print(f"\nRasio Default    : {target_pct[1]:.1f}%")
print(f"Rasio Non-Default: {target_pct[0]:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
colors = ['#2ECC71', '#E74C3C']
axes[0].bar(['Non-Default (0)', 'Default (1)'],
             target_counts.values,
             color=colors, edgecolor='black', alpha=0.8)
axes[0].set_title('Distribusi Loan Status', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Jumlah')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(target_pct.values, labels=['Non-Default', 'Default'],
             colors=colors, autopct='%1.1f%%', startangle=90,
             explode=[0, 0.05], shadow=True)
axes[1].set_title('Proporsi Target Variable', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# --- 4.4 Analisis Korelasi: Pendapatan vs Default ---
print("=" * 60)
print("ANALISIS KORELASI: PENDAPATAN vs DEFAULT")
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribusi pendapatan per status
for status, color, label in [(0, '#2ECC71', 'Non-Default'), (1, '#E74C3C', 'Default')]:
    subset = df_raw[df_raw['loan_status'] == status]['person_income']
    axes[0].hist(subset.clip(upper=200000), bins=50, alpha=0.6,
                  color=color, label=label, edgecolor='none')

axes[0].set_title('Distribusi Pendapatan per Loan Status', fontsize=13, fontweight='bold')
axes[0].set_xlabel('person_income (clipped at 200K)')
axes[0].set_ylabel('Frekuensi')
axes[0].legend()

# Box plot
df_raw.boxplot(column='person_income', by='loan_status', ax=axes[1],
                 patch_artist=True,
                 boxprops=dict(facecolor='lightblue', color='black'),
                 medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Boxplot Pendapatan per Loan Status', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Loan Status (0=Non-Default, 1=Default)')
axes[1].set_ylabel('person_income')
plt.suptitle('')  # Hilangkan judul otomatis dari boxplot

plt.tight_layout()
plt.show()

# Statistik ringkasan
print("\nRata-rata Pendapatan per Status:")
print(df_raw.groupby('loan_status')['person_income'].agg(['mean', 'median']))
print("\n💡 Insight: Nasabah dengan pendapatan lebih rendah cenderung memiliki risiko default lebih tinggi.")

In [ ]:
# --- 4.5 Analisis Kolom Kategorikal ---
print("=" * 60)
print("ANALISIS KOLOM KATEGORIKAL")
print("=" * 60)

categorical_cols = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    # Default rate per kategori
    default_rate = df_raw.groupby(col)['loan_status'].mean().sort_values(ascending=False)
    bars = axes[i].bar(default_rate.index, default_rate.values * 100,
                         color='#3498DB', edgecolor='black', alpha=0.8)
    axes[i].set_title(f'Default Rate per {col}', fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Default Rate (%)')
    axes[i].axhline(y=df_raw['loan_status'].mean() * 100, color='red',
                     linestyle='--', linewidth=1.5, label='Rata-rata Overall')
    axes[i].legend(fontsize=8)
    axes[i].tick_params(axis='x', rotation=15)

    # Tambah label nilai
    for bar, val in zip(bars, default_rate.values * 100):
        axes[i].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                      f'{val:.1f}%', ha='center', va='bottom', fontsize=8)

plt.suptitle('Default Rate per Kategori Fitur', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- 4.6 Correlation Matrix (Fitur Numerik) ---
print("=" * 60)
print("CORRELATION MATRIX")
print("=" * 60)

numeric_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df_raw[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
             mask=mask, center=0, square=True, ax=ax,
             annot_kws={'size': 9}, linewidths=0.5)
ax.set_title('Correlation Matrix - Fitur Numerik', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Korelasi terhadap target
print("\nKorelasi terhadap loan_status (target):")
print(corr_matrix['loan_status'].drop('loan_status').sort_values(ascending=False))

In [ ]:
# --- 4.7 Analisis Outlier ---
print("=" * 60)
print("ANALISIS OUTLIER")
print("=" * 60)

outlier_cols = ['person_age', 'person_income', 'person_emp_length', 'loan_amnt']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, col in zip(axes, outlier_cols):
    df_raw[col].dropna().plot(kind='box', ax=ax, patch_artist=True,
                                boxprops=dict(facecolor='#AED6F1', color='#1A5276'),
                                medianprops=dict(color='#E74C3C', linewidth=2),
                                flierprops=dict(marker='o', color='#E74C3C',
                                                alpha=0.5, markersize=3))
    ax.set_title(col, fontsize=10, fontweight='bold')

plt.suptitle('Deteksi Outlier - Box Plot', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# IQR method untuk deteksi outlier
for col in outlier_cols:
    q1, q3 = df_raw[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_outlier = ((df_raw[col] < q1 - 1.5*iqr) | (df_raw[col] > q3 + 1.5*iqr)).sum()
    print(f"   {col:<30} : {n_outlier:5,} outlier ({n_outlier/len(df_raw)*100:.1f}%)")

# **5. Data Preprocessing**

Berdasarkan hasil EDA, kita akan melakukan preprocessing dengan langkah-langkah berikut:

1. **Menangani Missing Values** – Imputasi median untuk `person_emp_length` dan `loan_int_rate`
2. **Encoding Kategorikal** – Label Encoding untuk `loan_grade`, `cb_person_default_on_file`; One-Hot Encoding untuk `person_home_ownership` dan `loan_intent`
3. **Standarisasi** – StandardScaler pada `person_income` dan `loan_amnt`
4. **Feature Engineering** – Tambah fitur rasio debt-to-income
5. **Split Dataset** – Train/Test split (80:20) dengan stratifikasi


In [ ]:
# --- 5.1 Copy Working DataFrame ---
df = df_raw.copy()
print(f"Shape sebelum preprocessing: {df.shape}")
print(f"Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

In [ ]:
# --- 5.2 Menangani Missing Values ---
# Gunakan median imputation pada kolom numerik yang memiliki missing values

NUMERIC_IMPUTE_COLS = ['person_emp_length', 'loan_int_rate']

for col in NUMERIC_IMPUTE_COLS:
    median_val = df[col].median()
    n_before = df[col].isnull().sum()
    df[col].fillna(median_val, inplace=True)
    print(f"✅ {col:<30}: Imputasi {n_before:,} nilai → median={median_val:.2f}")

print(f"\nMissing values setelah imputasi: {df.isnull().sum().sum()}")

In [ ]:
# --- 5.3 Feature Engineering ---
# Tambah fitur baru yang bermakna secara bisnis

df['debt_to_income_ratio'] = df['loan_amnt'] / (df['person_income'] + 1)  # +1 hindari div by zero
df['income_per_year_employed'] = df['person_income'] / (df['person_emp_length'] + 1)

print("✅ Fitur baru ditambahkan:")
print("   - debt_to_income_ratio      : loan_amnt / person_income")
print("   - income_per_year_employed  : person_income / person_emp_length")

In [ ]:
# --- 5.4 Encoding Variabel Kategorikal ---

# Label Encoding untuk variabel ordinal dan binary
LABEL_ENCODE_COLS = {
    'loan_grade': {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6},
    'cb_person_default_on_file': {'N': 0, 'Y': 1}
}
for col, mapping in LABEL_ENCODE_COLS.items():
    df[col] = df[col].map(mapping)
    print(f"✅ Label Encoded : {col:<35} → {df[col].unique().tolist()}")

# One-Hot Encoding untuk variabel nominal
OHE_COLS = ['person_home_ownership', 'loan_intent']
df = pd.get_dummies(df, columns=OHE_COLS, drop_first=False, dtype=int)
print(f"\n✅ One-Hot Encoded : {OHE_COLS}")
print(f"   Shape setelah OHE: {df.shape}")

# Tampilkan kolom baru dari OHE
new_cols = [c for c in df.columns if any(c.startswith(p) for p in OHE_COLS)]
print(f"   Kolom baru       : {new_cols}")

In [ ]:
# --- 5.5 Standarisasi Fitur Numerik ---

SCALE_COLS = ['person_income', 'loan_amnt', 'person_age',
              'person_emp_length', 'loan_int_rate', 'loan_percent_income',
              'cb_person_cred_hist_length', 'debt_to_income_ratio',
              'income_per_year_employed']

scaler = StandardScaler()
df[SCALE_COLS] = scaler.fit_transform(df[SCALE_COLS])

print("✅ StandardScaler diterapkan pada kolom numerik:")
for col in SCALE_COLS:
    print(f"   {col:<35}: mean={df[col].mean():.4f}, std={df[col].std():.4f}")

In [ ]:
# --- 5.6 Split Dataset ---

TARGET_COL = 'loan_status'
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✅ Dataset di-split dengan stratifikasi")
print(f"   X_train : {X_train.shape}")
print(f"   X_test  : {X_test.shape}")
print(f"   y_train : {y_train.shape} | Default rate: {y_train.mean():.1%}")
print(f"   y_test  : {y_test.shape}  | Default rate: {y_test.mean():.1%}")

In [ ]:
# --- 5.7 Simpan Hasil Preprocessing ---

os.makedirs('credit_risk_preprocessing', exist_ok=True)

# Gabung X dan y untuk simpan ke CSV
train_df = X_train.copy()
train_df[TARGET_COL] = y_train.values

test_df = X_test.copy()
test_df[TARGET_COL] = y_test.values

train_df.to_csv('credit_risk_preprocessing/credit_risk_train.csv', index=False)
test_df.to_csv('credit_risk_preprocessing/credit_risk_test.csv', index=False)

print("✅ Hasil preprocessing disimpan:")
print(f"   - credit_risk_preprocessing/credit_risk_train.csv ({len(train_df):,} baris, {len(train_df.columns)} kolom)")
print(f"   - credit_risk_preprocessing/credit_risk_test.csv  ({len(test_df):,} baris, {len(test_df.columns)} kolom)")
print(f"\nFitur akhir ({len(X.columns)} kolom):")
print(list(X.columns))

In [ ]:
# --- 5.8 Verifikasi Hasil Preprocessing ---

print("=" * 60)
print("VERIFIKASI HASIL PREPROCESSING")
print("=" * 60)

print("\n📊 Statistik Dataset Training (5 kolom pertama):")
print(train_df.iloc[:, :5].describe().round(3))

print("\n📋 Missing Values Setelah Preprocessing:")
total_missing = train_df.isnull().sum().sum() + test_df.isnull().sum().sum()
print(f"   Total missing values: {total_missing}")

print("\n✅ Preprocessing selesai! Data siap untuk pelatihan model.")